# RoBERTa aspect-based sentiment classifier

This notebook trains one RoBERTa model to predict every valid `(aspectCategory, polarity)` pair in a sentence. The target is multi-label because one sentence can describe several aspects with different sentiments. Data is split 80:10:10 by review `id`, so duplicate rows from the same review cannot leak across splits.

In [7]:
# Run once if these packages are not installed in the notebook environment.
!uv pip install -q pandas scikit-learn iterative-stratification datasets transformers accelerate torch

In [8]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import classification_report, f1_score
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
MODEL_NAME = "FacebookAI/roberta-base"
MAX_LENGTH = 256

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [9]:
# This works whether Jupyter starts in the repository root or notebooks/.
repo_root = Path.cwd()
if not (repo_root / "data" / "contest2_train.csv").exists():
    repo_root = repo_root.parent

df = pd.read_csv(repo_root / "data" / "contest2_train.csv")
required_columns = {"id", "text", "aspectCategory", "polarity"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

df = df.dropna(subset=list(required_columns)).copy()
df["id"] = df["id"].astype(str)
df["text"] = df["text"].astype(str)
df["joint_label"] = list(zip(df["aspectCategory"], df["polarity"]))

print(f"Rows: {len(df):,}; reviews: {df['id'].nunique():,}")
display(df.head())
display(pd.crosstab(df["aspectCategory"], df["polarity"], margins=True))

Rows: 3,156; reviews: 2,584


,id,text,aspectCategory,polarity,joint_label
0,3121,But the staff was so horrible to us.,service,negative,"(service, negative)"
1,2777,"To be completely fair, the only redeeming fact...",food,positive,"(food, positive)"
2,2777,"To be completely fair, the only redeeming fact...",anecdotes/miscellaneous,negative,"(anecdotes/miscellaneous, negative)"
3,1634,"The food is uniformly exceptional, with a very...",food,positive,"(food, positive)"
4,2534,Where Gabriela personaly greets you and recomm...,service,positive,"(service, positive)"


polarity,conflict,negative,neutral,positive,All
aspectCategory,,,,,
ambience,41,78,21,228,368
anecdotes/miscellaneous,24,176,285,471,956
food,57,182,69,743,1051
price,15,100,8,152,275
service,30,179,15,282,506
All,167,715,398,1876,3156


## Build multi-label examples and split 80:10:10

Rows sharing a review ID are kept together. Each unique text becomes one example whose label vector can contain multiple category–polarity pairs.

In [10]:
aspects = sorted(df["aspectCategory"].unique())
polarities = sorted(df["polarity"].unique())
label_pairs = [(aspect, polarity) for aspect in aspects for polarity in polarities]
label2id = {pair: index for index, pair in enumerate(label_pairs)}
id2label = {index: f"{aspect}::{polarity}" for index, (aspect, polarity) in enumerate(label_pairs)}

examples = (
    df.groupby(["id", "text"], as_index=False, sort=False)["joint_label"]
    .agg(lambda labels: sorted(set(labels)))
)

def encode_labels(pairs):
    target = np.zeros(len(label_pairs), dtype=np.float32)
    for pair in pairs:
        target[label2id[pair]] = 1.0
    return target.tolist()

examples["labels"] = examples["joint_label"].map(encode_labels)

# Each ID has exactly one text, so one row per ID lets iterative stratification
# preserve all 20 label frequencies without allowing review leakage.
if examples["id"].duplicated().any():
    raise ValueError("Expected one text per review ID before stratification.")
all_targets = np.asarray(examples["labels"].tolist(), dtype=np.int8)
outer_split = MultilabelStratifiedShuffleSplit(
    n_splits=1, test_size=0.20, random_state=SEED
)
train_idx, holdout_idx = next(outer_split.split(examples[["text"]], all_targets))
train_df = examples.iloc[train_idx].reset_index(drop=True)
holdout_df = examples.iloc[holdout_idx].reset_index(drop=True)

holdout_targets = np.asarray(holdout_df["labels"].tolist(), dtype=np.int8)
inner_split = MultilabelStratifiedShuffleSplit(
    n_splits=1, test_size=0.50, random_state=SEED
)
eval_idx, test_idx = next(inner_split.split(holdout_df[["text"]], holdout_targets))
eval_df = holdout_df.iloc[eval_idx].reset_index(drop=True)
test_df = holdout_df.iloc[test_idx].reset_index(drop=True)

split_ids = [set(part["id"]) for part in (train_df, eval_df, test_df)]
assert split_ids[0].isdisjoint(split_ids[1])
assert split_ids[0].isdisjoint(split_ids[2])
assert split_ids[1].isdisjoint(split_ids[2])

total = len(examples)
for name, part in [("train", train_df), ("eval", eval_df), ("test", test_df)]:
    print(f"{name:>5}: {len(part):4d} examples ({len(part) / total:.1%})")
print(f"Labels ({len(label_pairs)}): {list(id2label.values())}")

# Compare prevalence to verify that stratification preserved label balance.
prevalence = pd.DataFrame(
    {
        name: np.asarray(part["labels"].tolist()).mean(axis=0)
        for name, part in [("all", examples), ("train", train_df), ("eval", eval_df), ("test", test_df)]
    },
    index=list(id2label.values()),
)
display(prevalence.style.format("{:.2%}"))

train: 2074 examples (80.3%)
 eval:  254 examples (9.8%)
 test:  256 examples (9.9%)
Labels (20): ['ambience::conflict', 'ambience::negative', 'ambience::neutral', 'ambience::positive', 'anecdotes/miscellaneous::conflict', 'anecdotes/miscellaneous::negative', 'anecdotes/miscellaneous::neutral', 'anecdotes/miscellaneous::positive', 'food::conflict', 'food::negative', 'food::neutral', 'food::positive', 'price::conflict', 'price::negative', 'price::neutral', 'price::positive', 'service::conflict', 'service::negative', 'service::neutral', 'service::positive']


,all,train,eval,test
ambience::conflict,1.59%,1.59%,1.57%,1.56%
ambience::negative,3.02%,2.99%,3.15%,3.12%
ambience::neutral,0.81%,0.82%,0.79%,0.78%
ambience::positive,8.82%,8.78%,9.06%,8.98%
anecdotes/miscellaneous::conflict,0.93%,0.92%,1.18%,0.78%
anecdotes/miscellaneous::negative,6.81%,6.80%,7.09%,6.64%
anecdotes/miscellaneous::neutral,11.03%,10.99%,11.02%,11.33%
anecdotes/miscellaneous::positive,18.15%,18.08%,18.50%,18.36%
food::conflict,2.21%,2.22%,2.36%,1.95%
food::negative,7.04%,7.04%,7.09%,7.03%


In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def to_dataset(frame):
    dataset = Dataset.from_pandas(frame[["text", "labels"]], preserve_index=False)
    return dataset.map(
        lambda batch: tokenizer(
            batch["text"], truncation=True, max_length=MAX_LENGTH
        ),
        batched=True,
        remove_columns=["text"],
    )

train_dataset = to_dataset(train_df)
eval_dataset = to_dataset(eval_df)
test_dataset = to_dataset(test_df)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map: 100%|██████████| 256/256 [00:00<00:00, 55778.80 examples/s]


In [16]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_pairs),
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id={name: index for index, name in id2label.items()},
)

def sigmoid(logits):
    return 1.0 / (1.0 + np.exp(-logits))

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = (sigmoid(logits) >= 0.5).astype(int)
    labels = labels.astype(int)
    return {
        "micro_f1": f1_score(labels, predictions, average="micro", zero_division=0),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
        "exact_match": float(np.mean(np.all(labels == predictions, axis=1))),
    }

training_args = TrainingArguments(
    output_dir=str(repo_root / "artifacts" / "roberta_multitask"),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=16,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [17]:
train_result = trainer.train()
trainer.save_model()
tokenizer.save_pretrained(training_args.output_dir)
train_result.metrics

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Exact Match
1,No log,0.237305,0.000000,0.000000,0.000000
2,No log,0.173287,0.289406,0.038621,0.125984
3,No log,0.144958,0.480000,0.119363,0.291339
4,0.247300,0.124858,0.603922,0.226064,0.429134
5,0.247300,0.114235,0.654206,0.296860,0.507874
6,0.247300,0.108634,0.703833,0.343356,0.578740
7,0.247300,0.114362,0.697124,0.358130,0.590551
8,0.084800,0.105539,0.717949,0.385106,0.606299
9,0.084800,0.104528,0.720812,0.400952,0.614173
10,0.084800,0.101726,0.732773,0.418152,0.618110


{'train_runtime': 95.0293,
 'train_samples_per_second': 349.197,
 'train_steps_per_second': 21.888,
 'total_flos': 678308157620016.0,
 'train_loss': 0.102821444777342,
 'epoch': 16.0}

## Tune the decision threshold on evaluation data

The evaluation split selects the probability threshold. The test split remains untouched until the final measurement.

In [18]:
eval_output = trainer.predict(eval_dataset)
eval_probabilities = sigmoid(eval_output.predictions)
eval_labels = eval_output.label_ids.astype(int)

threshold_scores = []
for threshold in np.arange(0.10, 0.91, 0.05):
    predictions = (eval_probabilities >= threshold).astype(int)
    score = f1_score(eval_labels, predictions, average="micro", zero_division=0)
    threshold_scores.append((float(threshold), score))

best_threshold, best_eval_f1 = max(threshold_scores, key=lambda item: item[1])
print(f"Best threshold: {best_threshold:.2f}; eval micro-F1: {best_eval_f1:.4f}")

Best threshold: 0.50; eval micro-F1: 0.7492


In [20]:
test_output = trainer.predict(test_dataset)
test_probabilities = sigmoid(test_output.predictions)
test_labels = test_output.label_ids.astype(int)
test_predictions = (test_probabilities >= best_threshold).astype(int)

print("Joint aspect-polarity results")
print(classification_report(
    test_labels,
    test_predictions,
    target_names=list(id2label.values()),
    zero_division=0,
))

Joint aspect-polarity results
                                   precision    recall  f1-score   support

               ambience::conflict       0.00      0.00      0.00         4
               ambience::negative       0.60      0.38      0.46         8
                ambience::neutral       0.00      0.00      0.00         2
               ambience::positive       0.75      0.65      0.70        23
anecdotes/miscellaneous::conflict       0.00      0.00      0.00         2
anecdotes/miscellaneous::negative       0.50      0.41      0.45        17
 anecdotes/miscellaneous::neutral       0.68      0.59      0.63        29
anecdotes/miscellaneous::positive       0.80      0.83      0.81        47
                   food::conflict       0.25      0.20      0.22         5
                   food::negative       0.69      0.50      0.58        18
                    food::neutral       0.50      0.43      0.46         7
                   food::positive       0.88      0.82      0.85     

In [ ]:
# Also report category and sentiment independently by collapsing the joint labels.
aspect_indices = {aspect: [label2id[(aspect, p)] for p in polarities] for aspect in aspects}
polarity_indices = {polarity: [label2id[(a, polarity)] for a in aspects] for polarity in polarities}

def collapse_joint(matrix, index_groups):
    return np.column_stack([matrix[:, indices].max(axis=1) for indices in index_groups.values()])

for name, groups in [("Aspect category", aspect_indices), ("Polarity", polarity_indices)]:
    collapsed_true = collapse_joint(test_labels, groups)
    collapsed_pred = collapse_joint(test_predictions, groups)
    print(f"\n{name} results")
    print(classification_report(
        collapsed_true,
        collapsed_pred,
        target_names=list(groups),
        zero_division=0,
    ))

## Predict the unlabeled contest set

A sentence may produce several output rows. If no score reaches the tuned threshold, its highest-scoring pair is used so every input receives at least one prediction.

In [ ]:
contest_df = pd.read_csv(repo_root / "data" / "contest2_test.csv")
contest_dataset = Dataset.from_pandas(contest_df[["text"]], preserve_index=False).map(
    lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH),
    batched=True,
    remove_columns=["text"],
)
contest_probabilities = sigmoid(trainer.predict(contest_dataset).predictions)

prediction_rows = []
for row, probabilities in zip(contest_df.itertuples(index=False), contest_probabilities):
    selected = np.flatnonzero(probabilities >= best_threshold).tolist()
    if not selected:
        selected = [int(np.argmax(probabilities))]
    for label_index in selected:
        aspect, polarity = label_pairs[label_index]
        prediction_rows.append({
            "id": row.id,
            "text": row.text,
            "aspectCategory": aspect,
            "polarity": polarity,
            "score": float(probabilities[label_index]),
        })

predictions_df = pd.DataFrame(prediction_rows)
prediction_path = repo_root / "artifacts" / "roberta_multitask_predictions.csv"
prediction_path.parent.mkdir(parents=True, exist_ok=True)
predictions_df.to_csv(prediction_path, index=False)
print(f"Saved {len(predictions_df):,} predicted labels to {prediction_path}")
display(predictions_df.head(10))